## 🧠 Overview: Image Duplicate Detection Pipeline
This notebook demonstrates an efficient workflow for identifying and removing duplicate and near-duplicate images using a focused two-stage approach:
- **MD5 Hashing** – Identifies and eliminates exact (bitwise identical) duplicates  
- **FAISS with Deep Features** – Detects perceptual duplicates using embeddings from a pre-trained ResNet model and approximate nearest neighbor search

The pipeline processes images directly from Google Drive, performs deduplication, evaluates performance with precision/recall/F1 metrics, and outputs a cleaned folder containing unique image representatives.

⚙️ Step 1: Imports

In [ ]:
from pathlib import Path
from PIL import Image
import os
import cv2
import numpy as np
import torch
import faiss
from torchvision import transforms
from tqdm import tqdm
import timm
import hashlib
from colorama import Fore, Style, init
from functools import lru_cache
import psutil, time
import itertools
from pathlib import Path
from collections import defaultdict
from typing import List, Set, Tuple, Dict, Any
import matplotlib.pyplot as plt

🧮 Step 2: Ground Truth and Evaluation Utilities

In [ ]:
def _build_ground_truth(all_files: List[Path]) -> List[List[Path]]:
    gt_groups_dict = defaultdict(list)

    for file_path in all_files:
        parts = file_path.name.split('_')
        if len(parts) >= 2:
            group_id = parts[0]
            gt_groups_dict[group_id].append(file_path)
        else:
            gt_groups_dict[file_path.name].append(file_path)

    return list(gt_groups_dict.values())

def _get_pairs(groups: List[List[Path]]) -> Set[Tuple[Path, Path]]:
    all_pairs = set()
    for group in groups:
        for file1, file2 in itertools.combinations(group, 2):
            canonical_pair = tuple(sorted((file1, file2)))
            all_pairs.add(canonical_pair)
    return all_pairs

def calculate_metrics(predicted_groups: List[List[Path]],
                      all_processed_files: List[Path]) -> Dict[str, Any]:
    gt_groups = _build_ground_truth(all_processed_files)
    gt_pairs = _get_pairs(gt_groups)
    tp_plus_fn = len(gt_pairs)

    pred_pairs = _get_pairs(predicted_groups)
    tp_plus_fp = len(pred_pairs)

    tp_pairs = gt_pairs.intersection(pred_pairs)
    tp = len(tp_pairs)

    precision = tp / tp_plus_fp if tp_plus_fp > 0 else 0.0
    recall = tp / tp_plus_fn if tp_plus_fn > 0 else 0.0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

    return {
        "precision": precision,
        "recall": recall,
        "f1_score": f1
    }

🧠 Step 3: Model and Image Preprocessing

In [ ]:
# DEVICE setup
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

IMAGE_FOLDER = Path("img")
print(os.path.exists(IMAGE_FOLDER))
mem_hash = []
time_hash = []
metrics_hash = []

def get_model(model_name="resnet50"):
    model = timm.create_model(model_name, pretrained=True, num_classes = 0)
    model.to(DEVICE).eval() # move model to CPU/GPU, eval(): turn off dropout, batchnorm and use running stats
    return model

def get_transform():
    transform = transforms.Compose([
        # Normalize
        transforms.Resize((224,224)), # form of ResNet
        transforms.ToTensor(), # transform from PIL image (0-255) (H x W x C) ->torch.FloatTensor (0-1) (C x H x W)
        transforms.Normalize(mean = [0.485,0.456,0.406], std = [0.229,0.224,0.225]) # (x - mean) / std
    ])
    return transform

def read_image_cv2(file_path):
    img = cv2.imread(str(file_path))
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = Image.fromarray(img)
    return img

🔍 Step 4: Exact Duplicate Removal (MD5)

Identifies bitwise identical images using MD5 hashing and removes perfect duplicates.

In [ ]:
# MD5 - FILTER
def compute_md5(file_path):
    """Calculate MD5 hash for a image (bitwise)."""
    with open(file_path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

def remove_exact_duplicates(image_folder):
    image_paths = [f for f in image_folder.iterdir() if f.suffix.lower() in [".jpg", ".png", ".jpeg"]]
    hashes = {}
    duplicates = []

    image_files = list(Path(image_folder).glob("*.*"))
    total = len(image_files)

    for img_path in image_files:
        h = compute_md5(img_path)
        if h in hashes:
            duplicates.append(img_path)
        else:
            hashes[h] = img_path

    # delete duplicate
    for dup in duplicates:
        os.remove(dup)

    return list(hashes.values())

🧬 Step 5: Deep Feature Similarity with FAISS

Extracts image embeddings using a CNN and performs approximate nearest neighbor search with FAISS for semantic duplicate detection.

In [ ]:
def extract_embeddings_cv2(image_paths, batch_size = 32):
    # batch_size: number of image solving each time
    model = get_model()
    transforms = get_transform()
    embeddings = []
    valid_files = []

    for i in tqdm(range(0, len(image_paths), batch_size), desc = "Extract embeddings"):
        batch_files = image_paths[i:i + batch_size]
        batch_imgs = []

        for f in batch_files:
            img = read_image_cv2(f)
            if img is None:
                continue
            img_tensor = transforms(img)
            batch_imgs.append(img_tensor)
            valid_files.append(f)

        #batch_imgs empty -> skip
        if not batch_imgs:
            continue

        batch_tensor = torch.stack(batch_imgs).to(DEVICE)
        with torch.no_grad():
            batch_emb = model(batch_tensor)
            batch_emb = batch_emb.cpu().numpy()
            embeddings.append(batch_emb)

    embeddings = np.vstack(embeddings)
    return embeddings, valid_files

def faiss_group_duplicate(embeddings, filenames, distance_threshold = 0.6, k_neighbors = 10):
    # - embeddings: numpy array (n, d)
    # - filenames: list Path
    # - distance_threshold: cosine similarity threshold
    # - k_neighbors: neighbors checking

    faiss.normalize_L2(embeddings)
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)

    D, neighbors = index.search(embeddings, k = k_neighbors)

    visited = set()
    groups = []
    representatives = []

    for i, nbrs in enumerate(neighbors):
        if i in visited:
            continue
        group = [filenames[i]]
        visited.add(i)
        for j_idx, j in enumerate(nbrs[1:]): # skip itself
            if j in visited:
                continue
            sim = D[i, j_idx + 1]
            if sim >= distance_threshold:
                group.append(filenames[j])
                visited.add(j)
        # Choose representatives
        best_file = None
        best_score = -1
        for f in group:
            img = read_image_cv2(f)
            if img is None:
                continue
            w, h = img.size
            score = w * h
            if score > best_score:
                best_score = score
                best_file = f
        groups.append(group)
        representatives.append(best_file)
    return groups, representatives

📊 Step 6: Reporting, Logging, and Evaluation Metrics
Utility functions for summarizing results, timing, memory usage, and computing precision/recall/F1 metrics.

In [ ]:
# Config output
init(autoreset=True)

def print_summary(groups, representatives, all_processed_files, image_folder, metrics=None, total_initial=None, after_md5=None,
                  elapsed_time=None, mem_used=None, output_folder=""):
    print("\n" + "="*70)
    print(Fore.CYAN + Style.BRIGHT + "📊 DUPLICATE DETECTION SUMMARY".center(70))
    print("="*70)

    if total_initial is not None:
        print(f"{Fore.YELLOW}Total Initial Images: {Fore.WHITE}{total_initial}")
    if after_md5 is not None:
        removed = total_initial - after_md5 if total_initial is not None else "?"
        print(f"{Fore.YELLOW}Remaining After MD5 Filter: {Fore.WHITE}{after_md5} "
              f"({removed} removed)")

    print(f"{Fore.YELLOW}Total Clusters Found: {Fore.WHITE}{len(groups)}")
    print(f"{Fore.YELLOW}Total Representatives: {Fore.WHITE}{len(representatives)}")
    print(f"{Fore.YELLOW}Total Processed Images: {Fore.WHITE}{len(all_processed_files)}")
    print(f"{Fore.YELLOW}Cleaned Folder: {output_folder}")

    print("\n" + "="*70)
    print(Fore.CYAN + Style.BRIGHT + "📊 PERFORMANCE EVALUATION".center(70))
    print("="*70)

    if elapsed_time is not None:
        print(f"{Fore.GREEN}Processing Time: {Fore.WHITE}{elapsed_time:.2f} seconds")
    if mem_used is not None:
        print(f"{Fore.GREEN}Memory Used: {Fore.WHITE}{mem_used:.2f} MB")

    print("\n" + "="*70)
    print(Fore.CYAN + Style.BRIGHT + "📊 SOME CLUSTERS".center(70))
    print("="*70)
    for idx, group in enumerate(groups):
        if idx >= 5:
            break
        rep = representatives[idx]
        rep_name = rep.name if hasattr(rep, "name") else Path(rep).name
        print(f"{Fore.CYAN}Cluster {idx + 1:>3} {Fore.WHITE}| {len(group):>3} images | "
              f"Representative: {Fore.GREEN}{rep_name}")
    print("-"*70)

    if metrics:
        print("\n" + "="*70)
        print(Fore.CYAN + Style.BRIGHT + "📊 EVALUATION METRICS".center(70))
        print("="*70)
        print(f"{Fore.CYAN}Precision: {Fore.WHITE}{metrics.get('precision', 0)*100:.2f}%")
        print(f"{Fore.CYAN}Recall:    {Fore.WHITE}{metrics.get('recall', 0)*100:.2f}%")
        print(f"{Fore.CYAN}F1-Score:  {Fore.WHITE}{metrics.get('f1_score', 0)*100:.2f}%")
    print("="*70 + "\n")


def get_memory_usage():
    process = psutil.Process(os.getpid())
    mem_bytes = process.memory_info().rss
    return mem_bytes / (1024 ** 2)

🚀 Step 7: Main Pipeline

Orchestrates the entire deduplication workflow: exact duplicate removal, deep feature grouping, performance measurement, and output generation.

In [ ]:
def main(image_folder, method = "faiss", model_name="resnet50"):

    image_folder = Path(image_folder)
    image_paths = [f for f in image_folder.iterdir() if f.suffix.lower() in [".jpg", ".png", ".jpeg"]]
    total_initial = len(image_paths)

    unique_images = remove_exact_duplicates(image_folder)
    after_md5 = len(unique_images)

    #print(f"After remove absolute duplicates, remaining {len(unique_images)} valid images.\n")

    groups = []
    representatives = []
    all_processed_files= []

    # Timing
    start_time = time.time()
    mem_before = get_memory_usage()

    if method == "faiss":
        embeddings, valid_files = extract_embeddings_cv2(unique_images, batch_size=32)
        groups, representatives = faiss_group_duplicate(embeddings, valid_files, distance_threshold=0.6, k_neighbors=10)
        all_processed_files = valid_files

    # Result of timing

    elapsed_time = time.time() - start_time
    mem_after = get_memory_usage()
    mem_used = abs(mem_after - mem_before)

    output_folder = image_folder / ("Cleaned" + method)
    output_folder.mkdir(exist_ok=True)
    for f in representatives:
        img = cv2.imread(str(f))
        cv2.imwrite(str(output_folder / f.name), img)

    print(f"\nCleaned folder: {output_folder}")

    # Evaluation
    metrics = None
    if all_processed_files:
        metrics = calculate_metrics(groups, all_processed_files)
    else:
        print(Fore.RED + "\n⚠️ Cannot evaluate because no images were processed.")

    mem_hash.append(mem_used)
    time_hash.append(elapsed_time)
    metrics_hash.append(metrics)

    print_summary(groups, representatives, all_processed_files, image_folder, metrics, total_initial=total_initial, after_md5=after_md5,
                  elapsed_time=elapsed_time, mem_used=mem_used, output_folder=output_folder)
    return groups, representatives


🧪 Step 8: Run the Experiments

Run the pipeline using all three methods to compare their performance and clustering quality.

In [ ]:
#############################################
#                   MAIN                    #
groups, representatives = main(IMAGE_FOLDER)

#############################################

🖼️ Step 9: Visualization of Image Groups

In [ ]:
def resize_and_pad(image, target_w, target_h, bg_color=(255, 255, 255)):
    # bảo vệ nếu image là None
    if image is None:
        return np.full((target_h, target_w, 3), bg_color, dtype=np.uint8)

    original_h, original_w = image.shape[:2]
    if original_h == 0 or original_w == 0:
        return np.full((target_h, target_w, 3), bg_color, dtype=np.uint8)

    ratio = min(target_w / original_w, target_h / original_h)
    new_w = max(1, int(original_w * ratio))
    new_h = max(1, int(original_h * ratio))

    resized_img = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_AREA)
    padded_img = np.full((target_h, target_w, 3), bg_color, dtype=np.uint8)

    x_offset = (target_w - new_w) // 2
    y_offset = (target_h - new_h) // 2
    padded_img[y_offset:y_offset + new_h, x_offset:x_offset + new_w] = resized_img
    return padded_img

# helper đọc ảnh an toàn
def safe_imread(path):
    try:
        img = cv2.imread(str(path))
        if img is not None:
            # Chuyển từ BGR sang RGB cho matplotlib
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return img
    except Exception:
        return None

class ImageGroupVisualizer:
    def __init__(self, groups, representatives=None, screen_width=1920, n_groups=5):
        self.groups = groups
        self.representatives = representatives

        # Cấu hình hiển thị
        PADDING = 20
        n_total_groups = len(groups)
        self.display_indices = list(range(min(n_groups, n_total_groups)))
        self.n_display = len(self.display_indices)

        if self.n_display == 0:
            raise ValueError("Không có group nào để hiển thị.")

        # Kích thước
        CANVAS_W = int(screen_width)
        total_section_width = CANVAS_W - (PADDING * (self.n_display + 1))
        SECTION_W = max(100, total_section_width // self.n_display)
        THUMB_W = max(50, SECTION_W // 2)
        THUMB_H = THUMB_W
        REP_W = SECTION_W
        REP_H = max(THUMB_H * 2, 150)

        SECTION_H = REP_H + (THUMB_H * 2) + (PADDING * 2)
        CANVAS_H = SECTION_H + (PADDING * 2)

        # Lưu các thông số
        self.PADDING = PADDING
        self.SECTION_W = SECTION_W
        self.THUMB_W = THUMB_W
        self.THUMB_H = THUMB_H
        self.REP_W = REP_W
        self.REP_H = REP_H
        self.CANVAS_W = CANVAS_W
        self.CANVAS_H = CANVAS_H

        # Khởi tạo static board
        self.static_board = np.ones((CANVAS_H, CANVAS_W, 3), dtype=np.uint8) * 255

        # Vẽ representatives
        self._draw_representatives()

        # Setup matplotlib
        self.fig, self.ax = plt.subplots(figsize=(16, 6))
        plt.subplots_adjust(bottom=0.05)

        # Hiển thị
        self.update_display()

    def _draw_representatives(self):
        """Vẽ các representative images lên static board"""
        for display_pos, idx in enumerate(self.display_indices):
            group = self.groups[idx]
            x_start = self.PADDING + display_pos * (self.SECTION_W + self.PADDING)
            y_start = self.PADDING

            # Lấy representative
            rep_path = None
            if self.representatives is not None and len(self.representatives) > idx and self.representatives[idx] is not None:
                rep_path = self.representatives[idx]
            elif group:
                rep_path = group[0]

            rep_img_original = safe_imread(rep_path) if rep_path is not None else None
            rep_image = resize_and_pad(rep_img_original, self.REP_W, self.REP_H)

            y_end = y_start + self.REP_H
            x_end = x_start + self.REP_W
            self.static_board[y_start:y_end, x_start:x_end] = rep_image

    def update_display(self):
        """Hiển thị board với 4 thumbnails đầu của mỗi group"""
        dynamic_board = self.static_board.copy()

        for display_pos, idx in enumerate(self.display_indices):
            group = self.groups[idx]

            # Bỏ qua nếu không có thumbnail
            if not group or len(group) <= 1:
                continue

            col_x_start = self.PADDING + display_pos * (self.SECTION_W + self.PADDING)
            col_y_start = self.PADDING + self.REP_H + self.PADDING

            thumb_positions = [
                (col_y_start, col_x_start),
                (col_y_start, col_x_start + self.THUMB_W),
                (col_y_start + self.THUMB_H, col_x_start),
                (col_y_start + self.THUMB_H, col_x_start + self.THUMB_W)
            ]

            # Hiển thị 4 ảnh đầu tiên (index 1-4)
            for j in range(4):
                img_index = 1 + j  # bắt đầu từ index 1
                if img_index < len(group):
                    thumb_path = group[img_index]
                    img_original = safe_imread(thumb_path)
                    thumb_img = resize_and_pad(img_original, self.THUMB_W, self.THUMB_H)

                    y_pos, x_pos = thumb_positions[j]
                    y_end = y_pos + self.THUMB_H
                    x_end = x_pos + self.THUMB_W

                    dynamic_board[y_pos:y_end, x_pos:x_end] = thumb_img

        # Hiển thị
        self.ax.clear()
        self.ax.imshow(dynamic_board)
        self.ax.axis('off')
        plt.tight_layout()

    def show(self):
        """Hiển thị figure"""
        plt.show()

visualizer = ImageGroupVisualizer(
    groups=groups,  # biến groups đã có sẵn từ trước
    representatives=representatives,
    screen_width=1920,
    n_groups=5  # hiển thị 5 groups đầu tiên
)

visualizer.show()

📈 Final step: Visualize n groups

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from pathlib import Path
from collections import OrderedDict

flat = []
for gid, g in enumerate(groups):
    for p in g:
        if isinstance(p, (str, Path)):
            flat.append(Path(p))
        else:
            flat.append(Path(str(p)))

# preserve order and uniqueness
files = list(OrderedDict.fromkeys(flat))
if len(files) == 0:
    raise RuntimeError("Không có file nào trong 'groups' để vẽ.")

print(f"Preparing embeddings for {len(files)} images (from groups).")

label_map = {}
for gid, g in enumerate(groups):
    for p in g:
        label_map[str(Path(p))] = gid

embeddings, valid_files = extract_embeddings_cv2(files, batch_size=32)

valid_files = [Path(p) for p in valid_files]
print("Extracted embeddings:", embeddings.shape, "for", len(valid_files), "files.")

labels = np.array([label_map.get(str(p), -1) for p in valid_files])
unique_labels = np.unique(labels)
n_clusters = len([l for l in unique_labels if l != -1])
print(f"Labels found: {n_clusters} clusters (non-noise), with possible -1 (not in groups).")
dim = embeddings.shape[1]
pca_dim = 50 if dim > 50 else dim
if pca_dim < dim:
    pca = PCA(n_components=pca_dim, random_state=42)
    emb_pca = pca.fit_transform(embeddings)
else:
    emb_pca = embeddings.copy()

tsne = TSNE(n_components=2, init='pca', perplexity=30, random_state=42, learning_rate='auto', n_iter=1000)
emb_tsne = tsne.fit_transform(emb_pca)

plt.figure(figsize=(10,8))
cmap = plt.get_cmap('tab20', max(2, len(unique_labels)))
label_to_color = {}
idx_map = {lab: i for i, lab in enumerate(sorted([l for l in unique_labels if l != -1]))}
for lab in unique_labels:
    if lab == -1:
        label_to_color[lab] = (0.75,0.75,0.75,1.0)
    else:
        label_to_color[lab] = cmap(idx_map[lab] % cmap.N)

for (x,y), lab in zip(emb_tsne, labels):
    plt.scatter(x, y, s=28, color=label_to_color[lab], edgecolors='k', linewidths=0.18, alpha=0.9)

plt.title("t-SNE 2D (colored by groups from pipeline)")
plt.xlabel("t-SNE dim 1")
plt.ylabel("t-SNE dim 2")

from collections import Counter
cnts = Counter(labels)
from matplotlib.patches import Patch
legend_items = []
for lab in sorted(unique_labels):
    lbl = 'noise' if lab == -1 else f'group {int(lab)}'
    legend_items.append(Patch(facecolor=label_to_color[lab], edgecolor='k', label=f"{lbl} ({cnts[lab]})"))
plt.legend(handles=legend_items, bbox_to_anchor=(1.02,1), loc='upper left')

plt.tight_layout()
plt.show()
